In [ ]:
import os
import numpy as np
import plotly.graph_objects as go

# --- Updated signal function ---
def signal(data, samples, first='reference'):
    data_shape = data.shape[0]
    steps = int(data_shape / (2 * samples))

    if first.lower() == 'reference':
        reference_samples = np.mean(np.reshape(data[::2], (steps, samples)), axis=1)
        signal_samples = np.mean(np.reshape(data[1::2], (steps, samples)), axis=1)
    elif first.lower() == 'signal':
        signal_samples = np.mean(np.reshape(data[::2], (steps, samples)), axis=1)
        reference_samples = np.mean(np.reshape(data[1::2], (steps, samples)), axis=1)
    else:
        raise ValueError("`first` must be either 'reference' or 'signal'")

    signal_photon = signal_samples / reference_samples
    return signal_photon, reference_samples, signal_samples

# --- Wrapper function ---
def data_to_time_signal(data, samples, first='reference'):
    time = data['time_axis']
    signal_photon, reference_samples, signal_samples = signal(data['avg_data'], samples, first=first)
    return time, signal_photon, reference_samples, signal_samples

# --- Custom Plotly Template ---
fig_template = go.layout.Template()
fig_template.layout = {
    'template': 'simple_white+presentation',
    'autosize': False,
    'width': 800,
    'height': 600,
    'xaxis': {
        'ticks': 'inside',
        'mirror': 'ticks',
        'linewidth': 2.0,
        'tickwidth': 2.0,
        'ticklen': 6,
        'showline': True,
        'showgrid': False,
        'zerolinecolor': 'white',
    },
    'yaxis': {
        'ticks': 'inside',
        'mirror': 'ticks',
        'linewidth': 2.0,
        'tickwidth': 2.0,
        'ticklen': 6,
        'showline': True,
        'showgrid': False,
        'zerolinecolor': 'white'
    },
    'font': {
        'family': 'mathjax',
        'size': 22,
    }
}

# --- Base path ---
base_path = r"C:\Users\Dell\Desktop\Atanu_github\T1_measurements\codes\data\2025\May_12\arranging_datas_with_init_time\100us"

# --- Power folders ---
power_folders = ['5uW', '8uW', '15uW', '24uW']

# --- Initialize figure ---
fig = go.Figure()

# --- Process each power folder ---
for folder in power_folders:
    folder_path = os.path.join(base_path, folder)
    npz_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.npz')])
    
    if len(npz_files) < 3:
        print(f"Less than 3 datasets found in {folder_path}. Skipping.")
        continue

    signal_photon_list = []
    time_axis = None

    for npz_file in npz_files[:3]:  # Limit to 3 datasets
        npz_path = os.path.join(folder_path, npz_file)
        data = dict(np.load(npz_path))
        time_axis, signal_photon, _, _ = data_to_time_signal(data, 2000, first='signal')
        signal_photon_list.append(signal_photon[1:])  # skip the first element

    # Convert list to 2D array (shape: 3 x N)
    signal_array = np.vstack(signal_photon_list)

    # Calculate mean and std deviation
    mean_signal = np.mean(signal_array, axis=0)
    std_signal = np.std(signal_array, axis=0)

    fig.add_trace(go.Scatter(
        x=time_axis[1:],  # skip the first point
        y=mean_signal,
        mode='markers',
        name=f'{folder}',
        error_y=dict(
            type='data',
            array=std_signal,
            visible=True
        )
    ))

# --- Final layout ---
fig.update_layout(
    template=fig_template,
    title='Signal Photon vs Time for 100us Initialization',
    xaxis_title='Time',
    yaxis_title='Counts (T<sub>1</sub>)'
)

# --- Show and Save ---
fig.show()

html_output_path = os.path.join(base_path, "signal_vs_time_plot_with_errorbars.html")
fig.write_html(html_output_path)
print(f"Plot successfully saved to: {html_output_path}")


Plot successfully saved to: C:\Users\Dell\Desktop\Atanu_github\T1_measurements\codes\data\2025\May_12\arranging_datas_with_init_time\100us\signal_vs_time_plot.html
